# Tensor
了解tensor常用的处理方式：铺平，reshape，扩增维度，若干tensor拼接，添加非线性函数

## 铺平
是什么： 将一个任意维度的张量转换成一维向量。

### 目的：

在全连接层之前，将多维特征图（例如来自卷积层的 [batch, channels, height, width]）转换成一维向量。

方便进行后续的向量运算。

### 实现方式：

torch.flatten() / tf.reshape() / np.flatten()

view(-1) 或 reshape(-1) （在PyTorch中）

In [1]:
import torch

# 假设有一个来自卷积层的特征图
# [批大小, 通道数, 高, 宽]
x = torch.randn(4, 16, 7, 7) # 4张图片，16个通道，7x7分辨率
print("原始形状:", x.shape) # torch.Size([4, 16, 7, 7])

# 方法1: 从第一个维度开始铺平
x_flat = torch.flatten(x, start_dim=1)
print("铺平后形状:", x_flat.shape) # torch.Size([4, 784])
# 计算：16 * 7 * 7 = 784

# 方法2: 全部铺平成一维 (常用于全连接层输入)
x_flat_all = torch.flatten(x)
print("全部铺平形状:", x_flat_all.shape) # torch.Size([3136])
# 计算：4 * 16 * 7 * 7 = 3136

原始形状: torch.Size([4, 16, 7, 7])
铺平后形状: torch.Size([4, 784])
全部铺平形状: torch.Size([3136])


## Reshape
是什么： 改变张量的形状，但不改变其数据总量和内存布局。

### 目的：

将张量调整为网络层所需的输入形状。

在卷积层和全连接层之间进行形状转换。

进行张量的转置、拆分、合并等操作。

### 实现方式：

torch.reshape() / tensor.reshape()

torch.view() （PyTorch中，要求内存连续）

In [2]:
# 接上例
# 将铺平的特征图重新塑形为4D
x_reshaped = x_flat.reshape(4, 16, 7, 7)
print("Reshape回原状:", x_reshaped.shape) # torch.Size([4, 16, 7, 7])

# 改变为不同的形状
# 例如，将通道维度和空间维度合并
x_new = x.reshape(4, 16 * 7 * 7)
print("新形状:", x_new.shape) # torch.Size([4, 784])

# 或者改变为3D，例如用于RNN [batch, seq_len, features]
x_3d = x.reshape(4, 49, 16) # [batch, 7*7序列, 16个特征]
print("3D形状:", x_3d.shape) # torch.Size([4, 49, 16])

Reshape回原状: torch.Size([4, 16, 7, 7])
新形状: torch.Size([4, 784])
3D形状: torch.Size([4, 49, 16])


## 扩增维度
是什么： 在指定位置插入一个长度为1的维度。

### 目的：

为了满足某些操作的广播要求。

为没有批次维度的数据添加批次维度。

调整维度顺序以匹配层输入要求。

### 实现方式：

torch.unsqueeze(dim) / tensor.unsqueeze(dim)

tf.expand_dims()

使用 None 或 np.newaxis 索引

In [3]:
# 假设有一个单张图片数据 [通道, 高, 宽]
img = torch.randn(3, 224, 224)
print("原始图片形状:", img.shape) # torch.Size([3, 224, 224])

# 添加批次维度，使其成为 [batch, 通道, 高, 宽]
img_with_batch = img.unsqueeze(0) # 在维度0（最前面）添加
print("添加批次维度后:", img_with_batch.shape) # torch.Size([1, 3, 224, 224])

# 在末尾添加维度，例如为了某些操作
img_end = img.unsqueeze(-1) # -1 表示最后一个维度
print("末尾添加维度:", img_end.shape) # torch.Size([3, 224, 224, 1])

# 使用None索引（等同于unsqueeze）
img_batch_alt = img[None, :, :, :] # 在维度0添加
print("使用None索引:", img_batch_alt.shape) # torch.Size([1, 3, 224, 224])

原始图片形状: torch.Size([3, 224, 224])
添加批次维度后: torch.Size([1, 3, 224, 224])
末尾添加维度: torch.Size([3, 224, 224, 1])
使用None索引: torch.Size([1, 3, 224, 224])


## 若干Tensor拼接
是什么： 沿指定维度将多个张量连接在一起。

### 目的：

合并来自不同分支的特征（如在Inception模块或残差连接中）。

将多个样本组合成一个批次。

集成多个模型的输出。

### 实现方式：

torch.cat(tensors, dim=)： 沿现有维度拼接，要求除拼接维度外其他维度必须相同。

torch.stack(tensors, dim=)： 创建一个新维度来堆叠张量，要求所有张量的形状完全相同。

In [4]:
# 假设有两个特征图
feat1 = torch.randn(4, 32, 14, 14)
feat2 = torch.randn(4, 64, 14, 14)

# 1. torch.cat - 沿通道维度拼接
# 常用于特征融合
feat_combined = torch.cat([feat1, feat2], dim=1) # 沿通道维(1)拼接
print("Cat后形状:", feat_combined.shape) # torch.Size([4, 96, 14, 14])
# 通道数 32 + 64 = 96

# 2. torch.stack - 创建新维度堆叠
# 例如将多个模型的输出堆叠
output1 = torch.randn(4, 10)
output2 = torch.randn(4, 10)
output_stacked = torch.stack([output1, output2], dim=0)
print("Stack后形状:", output_stacked.shape) # torch.Size([2, 4, 10])
# 新维度0，大小为2（因为堆叠了两个张量）

# 错误示例：cat要求非cat维度必须相同
# feat_wrong = torch.cat([feat1, feat1], dim=0) # 可以，沿批次维
# feat_error = torch.cat([feat1, feat2], dim=2) # 错误！高和宽可能不同

Cat后形状: torch.Size([4, 96, 14, 14])
Stack后形状: torch.Size([2, 4, 10])


## 添加非线性函数
是什么： 对张量的每个元素应用一个非线性函数。

### 目的：

为神经网络引入非线性变换，使其能够学习复杂的模式和函数。

没有非线性，无论多少层神经网络都等价于一个单层线性模型。

### 常用非线性函数：

ReLU： f(x) = max(0, x)

优点： 计算简单，解决了梯度消失问题（在正区间）。

缺点： “Dead ReLU”问题，负数部分梯度为0。

```python
x = torch.tensor([-2., -1., 0., 1., 2.])
relu = torch.nn.ReLU()
output = relu(x) # tensor([0., 0., 0., 1., 2.])
```
Sigmoid： f(x) = 1 / (1 + exp(-x))

优点： 输出范围(0, 1)，适合二分类输出层。

缺点： 容易导致梯度消失，输出不是零中心的。

```python
sigmoid = torch.nn.Sigmoid()
output = sigmoid(x) # 值被压缩到0-1之间
```
Tanh： f(x) = (exp(x) - exp(-x)) / (exp(x) + exp(-x))

优点： 输出范围(-1, 1)，是零中心的。

缺点： 仍然存在梯度消失问题。

```python
tanh = torch.nn.Tanh()
output = tanh(x) # 值被压缩到-1到1之间
```
Leaky ReLU： f(x) = max(0.01x, x)

优点： 解决了Dead ReLU问题，负数部分有小的斜率。

```python
leaky_relu = torch.nn.LeakyReLU(0.01) # 0.01是负斜率
output = leaky_relu(x)
```

In [7]:
# 一个简单的网络片段，综合运用上述操作
class SimpleNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = torch.nn.Conv2d(3, 16, 3, padding=1)
        self.relu = torch.nn.ReLU()
        self.fc = torch.nn.Linear(16 * 32 * 32, 10) # 假设输入是32x32

    def forward(self, x):
        # x: [batch, 3, 32, 32]
        x = self.conv(x) # -> [batch, 16, 32, 32]
        x = self.relu(x) # 添加非线性
        x = torch.flatten(x, 1) # 铺平 -> [batch, 16*32*32]
        x = self.fc(x) # 全连接层 -> [batch, 10]
        return x

    print(x.shape)
    print(x)

torch.Size([4, 16, 7, 7])
tensor([[[[ 1.2425e+00,  7.9537e-01,  4.1497e-01,  ...,  1.0982e+00,
            9.2037e-01, -3.4084e-01],
          [-3.1398e-01,  2.0065e-01,  1.6681e-01,  ...,  3.8524e-01,
           -2.3763e+00, -2.1389e-02],
          [ 1.2451e+00,  1.1971e-01,  1.1580e-01,  ...,  7.6850e-01,
           -5.0506e-01, -2.1556e+00],
          ...,
          [-9.2347e-01,  2.3129e-01,  1.2785e+00,  ..., -7.0878e-01,
            1.5936e-01, -6.0910e-01],
          [ 3.7382e-01, -1.4698e+00,  1.2797e+00,  ..., -2.5073e-01,
           -3.8896e-01,  3.3076e-01],
          [-1.9262e+00, -1.0967e+00, -1.1134e+00,  ...,  2.3623e-01,
            1.0479e+00, -2.6036e-01]],

         [[-2.5028e-01, -5.8731e-01, -1.7208e+00,  ..., -3.0991e-01,
            3.3735e-01,  5.8304e-01],
          [ 2.3431e-01, -2.2237e-01,  2.2822e+00,  ...,  1.1238e+00,
           -3.8629e-02,  5.5702e-01],
          [ 5.9293e-01, -7.2332e-01, -9.8035e-01,  ..., -6.2710e-01,
            8.2451e-01, -1.5048e

# Pytorch 模型架构操作

## 基本结构
__init__(): 定义所有要学习的参数和固定操作

forward(): 定义数据如何流过这些层

永远不要直接调用 forward()，而是调用模型实例本身：output = model(input)

In [8]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()  # 必须调用父类初始化

        # 定义网络层
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

        # 全连接层
        self.fc1 = nn.Linear(64 * 8 * 8, 128)  # 假设输入是32x32，经过两次池化后为8x8
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # 定义前向传播：层之间的连接方式
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)

        # 铺平
        x = x.view(x.size(0), -1)  # 保持batch维度，其余铺平

        # 全连接层
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

## 模型封装和拼接

In [ ]:
## 直接组合
class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, 3),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3),
            nn.ReLU()
        )

    def forward(self, x):
        return self.conv_layers(x)

class Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 28 * 28, 128),  # 假设尺寸
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.fc_layers(x)

# 组合成大模型
class CompleteModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = FeatureExtractor()
        self.classifier = Classifier(num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # 铺平
        x = self.classifier(x)
        return x

In [ ]:
## nn.Sequential
class ComplexModel(nn.Module):
    def __init__(self):
        super().__init__()

        # 分支1
        self.branch1 = nn.Sequential(
            nn.Conv2d(3, 32, 3),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3),
            nn.ReLU()
        )

        # 分支2
        self.branch2 = nn.Sequential(
            nn.Conv2d(3, 16, 5),
            nn.ReLU(),
            nn.Conv2d(16, 16, 5),
            nn.ReLU()
        )

        # 合并层
        self.combine = nn.Sequential(
            nn.Conv2d(32 + 16, 64, 1),  # 合并两个分支
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),    # 全局平均池化
        )

        self.classifier = nn.Linear(64, 10)

    def forward(self, x):
        branch1_out = self.branch1(x)
        branch2_out = self.branch2(x)

        # 拼接两个分支的输出
        combined = torch.cat([branch1_out, branch2_out], dim=1)

        x = self.combine(combined)
        x = x.view(x.size(0), -1)  # 铺平
        x = self.classifier(x)
        return x

## 模型实例化、训练和预测

In [10]:
## 实例化
# 实例化模型
model = SimpleCNN(num_classes=10)

# 检查模型结构
print(model)

# 将模型移动到GPU（如果可用）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=4096, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [12]:
## 训练流程：损失信号和参数更新
import torch.optim as optim
from torch.utils.data import DataLoader

# 1. 准备数据
# 假设我们有 train_loader 和 val_loader
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 2. 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()  # 分类任务常用交叉熵损失
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam优化器

# 3. 训练循环
def train_model(model, train_loader, val_loader, epochs=10):
    model.train()  # 设置为训练模式（影响Dropout、BatchNorm等）

    for epoch in range(epochs):
        running_loss = 0.0

        for i, (inputs, labels) in enumerate(train_loader):
            # 将数据移动到设备
            inputs, labels = inputs.to(device), labels.to(device)

            # 梯度清零 - 非常重要！
            optimizer.zero_grad()

            # 前向传播
            outputs = model(inputs)

            # 计算损失
            loss = criterion(outputs, labels)

            # 反向传播 - 计算梯度
            loss.backward()

            # 参数更新 - 使用优化器更新权重
            optimizer.step()

            running_loss += loss.item()

            if i % 100 == 99:  # 每100个batch打印一次
                print(f'Epoch {epoch+1}, Batch {i+1}, Loss: {running_loss/100:.4f}')
                running_loss = 0.0

        # 每个epoch结束后在验证集上评估
        validate_model(model, val_loader)

def validate_model(model, val_loader):
    model.eval()  # 设置为评估模式
    correct = 0
    total = 0

    with torch.no_grad():  # 禁用梯度计算，节省内存
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Validation Accuracy: {accuracy:.2f}%')
    model.train()  # 改回训练模式

In [13]:
import numpy as np


## 预测
def predict(model, input_data):
    model.eval()  # 设置为评估模式

    with torch.no_grad():  # 不计算梯度
        if isinstance(input_data, np.ndarray):
            input_data = torch.from_numpy(input_data).float()

        input_data = input_data.to(device)

        # 前向传播
        outputs = model(input_data)

        # 获取预测结果
        if outputs.shape[1] > 1:  # 多分类
            probabilities = torch.softmax(outputs, dim=1)
            predicted_class = torch.argmax(probabilities, dim=1)
            return predicted_class.cpu().numpy(), probabilities.cpu().numpy()
        else:  # 二分类或回归
            return outputs.cpu().numpy()

# 使用示例
# predictions, probs = predict(model, test_data)

## 模型保存和加载

In [ ]:
# 保存完整模型（包括结构和参数）
torch.save(model, 'complete_model.pth')

# 加载完整模型
loaded_model = torch.load('complete_model.pth')
loaded_model.eval()

# 只保存模型参数（推荐方式）
torch.save(model.state_dict(), 'model_weights.pth')

# 加载模型参数（需要先创建模型结构）
model = SimpleCNN(num_classes=10)
model.load_state_dict(torch.load('model_weights.pth'))
model.eval()

In [ ]:
# 假设我们有一个预训练模型，想加载部分权重
def load_partial_weights(model, pretrained_path):
    # 加载预训练权重
    pretrained_dict = torch.load(pretrained_path)

    # 当前模型的权重
    model_dict = model.state_dict()

    # 1. 过滤掉不匹配的键（尺寸不匹配的层）
    pretrained_dict = {k: v for k, v in pretrained_dict.items()
                      if k in model_dict and model_dict[k].shape == v.shape}

    # 2. 更新当前模型的权重
    model_dict.update(pretrained_dict)

    # 3. 加载权重
    model.load_state_dict(model_dict)

    print(f"Loaded {len(pretrained_dict)}/{len(model_dict)} layers")
    return model

# 使用示例
# model = load_partial_weights(model, 'pretrained_weights.pth')

In [ ]:
## 训练时冻结某些参数
def setup_model_with_frozen_layers():
    model = ComplexModel()

    # 冻结特征提取层（前几层）
    for name, param in model.named_parameters():
        if 'branch1' in name or 'branch2' in name:  # 冻结分支
            param.requires_grad = False
            print(f"Frozen: {name}")
        else:
            print(f"Trainable: {name}")

    # 优化器只对需要梯度的参数进行优化
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.001
    )

    return model, optimizer

# 或者更精细的控制
def fine_tune_model(pretrained_model, num_new_classes):
    # 冻结所有参数
    for param in pretrained_model.parameters():
        param.requires_grad = False

    # 只解冻最后几层
    for param in pretrained_model.combine.parameters():
        param.requires_grad = True

    # 替换分类器（新任务）
    pretrained_model.classifier = nn.Linear(64, num_new_classes)

    # 新分类器的参数默认 requires_grad=True
    return pretrained_model

# 检查哪些层被冻结
def check_frozen_layers(model):
    for name, param in model.named_parameters():
        print(f"{name}: requires_grad = {param.requires_grad}")

In [ ]:
# 分阶段解冻训练
def staged_training(model, train_loader, epochs_per_stage=5):
    # 阶段1：只训练分类器
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

    print("Stage 1: Training classifier only")
    train_model(model, train_loader, epochs=epochs_per_stage)

    # 阶段2：解冻最后几个卷积层
    for name, param in model.named_parameters():
        if 'combine' in name or 'branch1.3' in name or 'branch2.3' in name:
            param.requires_grad = True

    print("Stage 2: Training last few conv layers + classifier")
    train_model(model, train_loader, epochs=epochs_per_stage)

    # 阶段3：解冻所有层
    for param in model.parameters():
        param.requires_grad = True

    print("Stage 3: Training all layers")
    train_model(model, train_loader, epochs=epochs_per_stage)

## Tensorboard

In [15]:
import torch
from tensorboardX import SummaryWriter
import numpy as np
from datetime import datetime

class TensorBoardLogger:
    """TensorBoard 日志记录器"""
    def __init__(self, log_dir=None):
        if log_dir is None:
            # 使用时间戳创建唯一的日志目录
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            log_dir = f"runs/experiment_{timestamp}"

        self.writer = SummaryWriter(log_dir)
        self.log_dir = log_dir
        print(f"TensorBoard logging to: {log_dir}")

    def log_scalar(self, tag, value, step):
        """记录标量值（损失、准确率等）"""
        self.writer.add_scalar(tag, value, step)

    def log_scalars(self, main_tag, tag_scalar_dict, step):
        """记录多个标量值"""
        self.writer.add_scalars(main_tag, tag_scalar_dict, step)

    def log_histogram(self, tag, values, step):
        """记录权重/梯度的分布"""
        self.writer.add_histogram(tag, values, step)

    def log_model_graph(self, model, input_tensor):
        """记录模型计算图"""
        self.writer.add_graph(model, input_tensor)

    def log_images(self, tag, images, step, dataformats='NCHW'):
        """记录图像"""
        self.writer.add_images(tag, images, step, dataformats=dataformats)

    def log_embedding(self, features, metadata=None, label_img=None, step=None):
        """记录嵌入向量（用于降维可视化）"""
        self.writer.add_embedding(features, metadata=metadata, label_img=label_img, global_step=step)

    def close(self):
        """关闭写入器"""
        self.writer.close()

# 修改训练器类以集成 TensorBoard
class ModelTrainerWithTensorBoard:
    def __init__(self, model, device, log_dir=None):
        self.model = model
        self.device = device
        self.logger = TensorBoardLogger(log_dir)

        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []
        self.learning_rates = []

    def train_epoch(self, train_loader, criterion, optimizer, epoch):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(self.device), targets.to(self.device)
            targets = targets.long()

            optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            # 记录每个batch的损失
            if batch_idx % 10 == 0:  # 每10个batch记录一次
                step = epoch * len(train_loader) + batch_idx
                self.logger.log_scalar('Loss/train_batch', loss.item(), step)

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100. * correct / total

        return epoch_loss, epoch_acc

    def validate(self, val_loader, criterion, epoch):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                targets = targets.long()
                outputs = self.model(inputs)
                loss = criterion(outputs, targets)

                running_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()

        epoch_loss = running_loss / len(val_loader)
        epoch_acc = 100. * correct / total

        return epoch_loss, epoch_acc

    def train(self, train_loader, val_loader, criterion, optimizer,
              scheduler=None, epochs=25, log_weights_freq=5):
        """完整训练过程，集成 TensorBoard 日志记录"""

        # 记录模型图
        sample_input, _ = next(iter(train_loader))
        sample_input = sample_input.to(self.device)
        self.logger.log_model_graph(self.model, sample_input)

        print("Starting training with TensorBoard logging...")

        for epoch in range(epochs):
            # 训练
            train_loss, train_acc = self.train_epoch(train_loader, criterion, optimizer, epoch)

            # 验证
            val_loss, val_acc = self.validate(val_loader, criterion, epoch)

            # 学习率调度
            current_lr = optimizer.param_groups[0]['lr']
            if scheduler:
                scheduler.step()

            # 记录指标
            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)
            self.train_accuracies.append(train_acc)
            self.val_accuracies.append(val_acc)
            self.learning_rates.append(current_lr)

            # 记录到 TensorBoard
            self.logger.log_scalar('Loss/train', train_loss, epoch)
            self.logger.log_scalar('Loss/val', val_loss, epoch)
            self.logger.log_scalar('Accuracy/train', train_acc, epoch)
            self.logger.log_scalar('Accuracy/val', val_acc, epoch)
            self.logger.log_scalar('Learning_rate', current_lr, epoch)

            # 记录权重和梯度的直方图（每隔几个epoch记录一次）
            if epoch % log_weights_freq == 0:
                for name, param in self.model.named_parameters():
                    self.logger.log_histogram(f'Weights/{name}', param, epoch)
                    if param.grad is not None:
                        self.logger.log_histogram(f'Gradients/{name}', param.grad, epoch)

            print(f'Epoch: {epoch+1:02d}/{epochs} | '
                  f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | '
                  f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | '
                  f'LR: {current_lr:.6f}')

        # 记录最终模型权重
        for name, param in self.model.named_parameters():
            self.logger.log_histogram(f'Final_Weights/{name}', param, epochs)

        print("Training completed!")
        self.logger.close()

        return self.logger.log_dir

主函数

In [16]:
def main_with_tensorboard():
    # 参数设置
    config = {
        'batch_size': 64,
        'epochs': 30,
        'learning_rate': 0.001,
        'weight_decay': 1e-4,
        'img_size': 32,
        'num_classes': 10
    }

    # 1. 准备数据
    print("Preparing data...")
    data_processor = DataProcessor(n_samples=2000, img_size=config['img_size'],
                                 n_classes=config['num_classes'])
    train_loader, test_loader = data_processor.prepare_dataloaders(
        batch_size=config['batch_size']
    )

    # 2. 创建模型和训练器（带TensorBoard）
    model = CompleteCNN(
        in_channels=3,
        img_size=config['img_size'],
        num_classes=config['num_classes']
    ).to(device)

    trainer = ModelTrainerWithTensorBoard(model, device)

    # 3. 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    # 4. 训练模型
    log_dir = trainer.train(
        train_loader, test_loader, criterion, optimizer,
        scheduler=scheduler, epochs=config['epochs']
    )

    print(f"\nTraining completed! To view TensorBoard, run:")
    print(f"tensorboard --logdir={log_dir}")
    print("Then open http://localhost:6006 in your browser")

    return log_dir

# 运行带TensorBoard的训练
if __name__ == "__main__":
    log_dir = main_with_tensorboard()

Preparing data...


NameError: name 'DataProcessor' is not defined

启动tensorboard

In [ ]:
# 在终端中运行（在代码所在目录）
tensorboard --logdir=runs

# 或者指定具体目录
tensorboard --logdir=path/to/your/logs

# 指定端口（如果默认6006被占用）
tensorboard --logdir=runs --port=6007

wandb

In [ ]:
import wandb
import matplotlib.pyplot as plt

class WandBLogger:
    """Weights & Biases 日志记录器"""
    def __init__(self, project_name="my_project", config=None):
        self.run = wandb.init(
            project=project_name,
            config=config,
            magic=True
        )

    def log_metrics(self, metrics, step=None):
        """记录指标"""
        if step is not None:
            metrics['step'] = step
        wandb.log(metrics)

    def watch_model(self, model, criterion=None, log='gradients', log_freq=100):
        """监控模型权重和梯度"""
        wandb.watch(model, criterion, log=log, log_freq=log_freq)

    def log_histogram(self, tag, values, step=None):
        """记录直方图"""
        wandb.log({tag: wandb.Histogram(values.cpu().detach().numpy())}, step=step)

    def log_images(self, tag, images, step=None):
        """记录图像"""
        wandb.log({tag: [wandb.Image(img) for img in images]}, step=step)

    def log_model(self, model, input_array):
        """记录模型"""
        wandb.log({"model": wandb.Model(model, input_array)})

    def finish(self):
        """结束运行"""
        wandb.finish()

class ModelTrainerWithWandB:
    def __init__(self, model, device, project_name, config):
        self.model = model
        self.device = device
        self.logger = WandBLogger(project_name, config)

        # 监控模型
        self.logger.watch_model(model, log='all', log_freq=100)

        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []

    def train(self, train_loader, val_loader, criterion, optimizer,
              scheduler=None, epochs=25):

        print("Starting training with Weights & Biases logging...")

        for epoch in range(epochs):
            # 训练
            train_loss, train_acc = self.train_epoch(train_loader, criterion, optimizer, epoch)

            # 验证
            val_loss, val_acc = self.validate(val_loader, criterion, epoch)

            # 学习率调度
            current_lr = optimizer.param_groups[0]['lr']
            if scheduler:
                scheduler.step()

            # 记录指标
            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)
            self.train_accuracies.append(train_acc)
            self.val_accuracies.append(val_acc)

            # 记录到 wandb
            self.logger.log_metrics({
                'epoch': epoch,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'train_accuracy': train_acc,
                'val_accuracy': val_acc,
                'learning_rate': current_lr
            }, step=epoch)

            print(f'Epoch: {epoch+1:02d}/{epochs} | '
                  f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | '
                  f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')

        # 创建并保存训练历史图
        self._create_training_plots()

        print("Training completed!")
        self.logger.finish()

    def _create_training_plots(self):
        """创建训练历史图并记录到wandb"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

        # 损失曲线
        ax1.plot(self.train_losses, label='Train Loss')
        ax1.plot(self.val_losses, label='Val Loss')
        ax1.set_title('Training and Validation Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True)

        # 准确率曲线
        ax2.plot(self.train_accuracies, label='Train Accuracy')
        ax2.plot(self.val_accuracies, label='Val Accuracy')
        ax2.set_title('Training and Validation Accuracy')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy (%)')
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()

        # 记录图表到wandb
        self.logger.log_metrics({'training_history': wandb.Image(fig)})
        plt.close()

# 使用 wandb 的主函数
def main_with_wandb():
    # 配置参数
    config = {
        'batch_size': 64,
        'epochs': 30,
        'learning_rate': 0.001,
        'weight_decay': 1e-4,
        'img_size': 32,
        'num_classes': 10,
        'optimizer': 'Adam',
        'architecture': 'CompleteCNN'
    }

    # 准备数据
    data_processor = DataProcessor(n_samples=2000, img_size=config['img_size'],
                                 n_classes=config['num_classes'])
    train_loader, test_loader = data_processor.prepare_dataloaders(
        batch_size=config['batch_size']
    )

    # 创建模型和训练器
    model = CompleteCNN(
        in_channels=3,
        img_size=config['img_size'],
        num_classes=config['num_classes']
    ).to(device)

    trainer = ModelTrainerWithWandB(model, device, "my_cnn_project", config)

    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    # 训练
    trainer.train(
        train_loader, test_loader, criterion, optimizer,
        scheduler=scheduler, epochs=config['epochs']
    )

mlflow

In [ ]:
import mlflow
import mlflow.pytorch

class MLflowLogger:
    """MLflow 日志记录器"""
    def __init__(self, experiment_name="default"):
        mlflow.set_experiment(experiment_name)

    def start_run(self, run_name=None):
        """开始运行"""
        mlflow.start_run(run_name=run_name)

    def log_params(self, params):
        """记录超参数"""
        mlflow.log_params(params)

    def log_metrics(self, metrics, step=None):
        """记录指标"""
        mlflow.log_metrics(metrics, step=step)

    def log_model(self, model, model_name):
        """记录模型"""
        mlflow.pytorch.log_model(model, model_name)

    def log_artifact(self, local_path):
        """记录文件"""
        mlflow.log_artifact(local_path)

    def end_run(self):
        """结束运行"""
        mlflow.end_run()

# 在训练循环中使用
def train_with_mlflow():
    with mlflow.start_run():
        # 记录超参数
        mlflow.log_params(config)

        # 训练过程...
        for epoch in range(epochs):
            # ... 训练代码 ...

            # 记录指标
            mlflow.log_metrics({
                'train_loss': train_loss,
                'val_loss': val_loss,
                'train_accuracy': train_acc,
                'val_accuracy': val_acc
            }, step=epoch)

        # 保存模型
        mlflow.pytorch.log_model(model, "model")

记录器

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

class SimpleLogger:
    """简单的CSV和JSON记录器"""
    def __init__(self, log_dir="logs"):
        import os
        self.log_dir = log_dir
        os.makedirs(log_dir, exist_ok=True)

        self.metrics = {
            'train_loss': [], 'val_loss': [],
            'train_acc': [], 'val_acc': [],
            'learning_rate': []
        }

    def log_epoch(self, train_loss, val_loss, train_acc, val_acc, lr, epoch):
        """记录每个epoch的指标"""
        self.metrics['train_loss'].append(train_loss)
        self.metrics['val_loss'].append(val_loss)
        self.metrics['train_acc'].append(train_acc)
        self.metrics['val_acc'].append(val_acc)
        self.metrics['learning_rate'].append(lr)

    def save_metrics(self):
        """保存指标到文件"""
        # 保存为JSON
        with open(f"{self.log_dir}/metrics.json", 'w') as f:
            json.dump(self.metrics, f, indent=2)

        # 保存为CSV
        df = pd.DataFrame(self.metrics)
        df.to_csv(f"{self.log_dir}/metrics.csv", index_label='epoch')

        print(f"Metrics saved to {self.log_dir}/")

    def plot_training_history(self, save_path=None):
        """绘制训练历史"""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 8))

        epochs = range(1, len(self.metrics['train_loss']) + 1)

        # 损失曲线
        ax1.plot(epochs, self.metrics['train_loss'], 'b-', label='Train Loss')
        ax1.plot(epochs, self.metrics['val_loss'], 'r-', label='Val Loss')
        ax1.set_title('Training and Validation Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True)

        # 准确率曲线
        ax2.plot(epochs, self.metrics['train_acc'], 'b-', label='Train Accuracy')
        ax2.plot(epochs, self.metrics['val_acc'], 'r-', label='Val Accuracy')
        ax2.set_title('Training and Validation Accuracy')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy (%)')
        ax2.legend()
        ax2.grid(True)

        # 学习率曲线
        ax3.plot(epochs, self.metrics['learning_rate'], 'g-')
        ax3.set_title('Learning Rate')
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('Learning Rate')
        ax3.grid(True)

        # 损失-准确率关系
        ax4.scatter(self.metrics['train_loss'], self.metrics['train_acc'],
                   alpha=0.5, label='Train', c='blue')
        ax4.scatter(self.metrics['val_loss'], self.metrics['val_acc'],
                   alpha=0.5, label='Val', c='red')
        ax4.set_title('Loss vs Accuracy')
        ax4.set_xlabel('Loss')
        ax4.set_ylabel('Accuracy (%)')
        ax4.legend()
        ax4.grid(True)

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')

        plt.show()